In [ ]:
from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

from sklearn.metrics import f1_score, roc_auc_score, average_precision_score
from sklearn.metrics import precision_recall_curve, accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV

from imblearn.combine import SMOTETomek
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

In [ ]:
from mlflow.tracking import MlflowClient

In [ ]:
import mlflow

In [ ]:
mlflow.__version__

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

smote_tomek = SMOTETomek(random_state=42)
rus = RandomUnderSampler()

x_train, y_train = smote_tomek.fit_resample(x_train, y_train)
x_train, y_train = rus.fit_resample(x_train, y_train)

In [ ]:
def show_result(model):
    pred_proba = model.predict_proba(x_test)[:, 1]

    precision, recall, thresholds = precision_recall_curve(y_test, pred_proba)
    f1_scores = 2 * (precision[:-1] * recall[:-1]) / (precision[:-1] + recall[:-1] + 1e-10)
    best_threshold = thresholds[np.argmax(f1_scores)]

    print(f"Оптимальный порог: {best_threshold:.4f}")

    pred_binary = (pred_proba >= best_threshold).astype(int)

    F1 = f1_score(y_test, pred_binary)
    ROC_AUC = roc_auc_score(y_test, pred_proba)
    pr_auc = average_precision_score(y_test, pred_proba)
    conf_matrix = confusion_matrix(y_test, pred_binary)

    print(f'F1 Score: {F1}')
    print(f'ROC AUC: {ROC_AUC}')
    print(f"PR AUC: {pr_auc}")
    print(f'Confusion matrix: \n {conf_matrix}')

In [ ]:
model = CatBoostClassifier(loss_function='Logloss', auto_class_weights='Balanced', early_stopping_rounds=15, verbose=0, random_state=42)

In [ ]:
param_list = {
    'iterations': [300, 350, 400, 500],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.8, 0.9, 1.0],
    'depth': [6, 8, 10],
    'l2_leaf_reg': [1, 3, 5],
    'rsm': [0.8, 0.9, 1.0],
}

In [ ]:
CV = GridSearchCV(estimator=model, 
                        param_grid=param_list, 
                        cv=3, 
                        verbose=3, 
                        scoring='average_precision')
CV.fit(x_train, y_train)

Fitting 3 folds for each of 972 candidates, totalling 2916 fits
[CV 1/3] END depth=6, iterations=300, l2_leaf_reg=1, learning_rate=0.05, rsm=0.8, subsample=0.8;, score=0.968 total time=   1.0s
[CV 2/3] END depth=6, iterations=300, l2_leaf_reg=1, learning_rate=0.05, rsm=0.8, subsample=0.8;, score=0.963 total time=   0.8s
[CV 3/3] END depth=6, iterations=300, l2_leaf_reg=1, learning_rate=0.05, rsm=0.8, subsample=0.8;, score=0.955 total time=   0.6s
[CV 1/3] END depth=6, iterations=300, l2_leaf_reg=1, learning_rate=0.05, rsm=0.8, subsample=0.9;, score=0.966 total time=   0.6s
[CV 2/3] END depth=6, iterations=300, l2_leaf_reg=1, learning_rate=0.05, rsm=0.8, subsample=0.9;, score=0.961 total time=   0.6s
[CV 3/3] END depth=6, iterations=300, l2_leaf_reg=1, learning_rate=0.05, rsm=0.8, subsample=0.9;, score=0.957 total time=   0.6s
[CV 1/3] END depth=6, iterations=300, l2_leaf_reg=1, learning_rate=0.05, rsm=0.8, subsample=1.0;, score=0.968 total time=   0.6s
[CV 2/3] END depth=6, iterations=

GridSearchCV(cv=3,
             estimator=<catboost.core.CatBoostClassifier object at 0x0000020207C1BF50>,
             param_grid={'depth': [6, 8, 10],
                         'iterations': [300, 350, 400, 500],
                         'l2_leaf_reg': [1, 3, 5],
                         'learning_rate': [0.05, 0.1, 0.15],
                         'rsm': [0.8, 0.9, 1.0], 'subsample': [0.8, 0.9, 1.0]},
             scoring='average_precision', verbose=3)

In [ ]:
print(CV.best_params_)

{'depth': 6, 'iterations': 500, 'l2_leaf_reg': 3, 'learning_rate': 0.1, 'rsm': 0.9, 'subsample': 1.0}


In [ ]:
model = CV.best_estimator_
show_result(model)

Оптимальный порог: 0.0940
F1 Score: 0.4027777777777778
ROC AUC: 0.6829184253596347
PR AUC: 0.32695474907799993
Confusion matrix: 
 [[179  68]
 [ 18  29]]


In [ ]:
model2 = XGBClassifier(device='cuda', verbosity=0)

In [ ]:
model2.fit(x_train, y_train)
show_result(model2)

Оптимальный порог: 0.2135
F1 Score: 0.4107142857142857
ROC AUC: 0.6922215522439485
PR AUC: 0.3441651501562006
Confusion matrix: 
 [[205  42]
 [ 24  23]]


In [ ]:
model3 = RandomForestClassifier(verbose=0, class_weight='balanced', random_state=42)

In [ ]:
param_list = {
    'n_estimators': [100, 200, 300, 400],
    'max_depth': [6, 8, 10, 12],
    'min_samples_split': [2, 3, 4, 5],
    'min_samples_leaf': [1, 2, 3, 4, 5],
}

In [ ]:
CV = GridSearchCV(estimator=model3, 
                        param_grid=param_list, 
                        cv=3, 
                        verbose=3, 
                        scoring='average_precision')
CV.fit(x_train, y_train)

Fitting 3 folds for each of 320 candidates, totalling 960 fits
[CV 1/3] END max_depth=6, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.922 total time=   0.1s
[CV 2/3] END max_depth=6, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.927 total time=   0.1s
[CV 3/3] END max_depth=6, min_samples_leaf=1, min_samples_split=2, n_estimators=100;, score=0.910 total time=   0.1s
[CV 1/3] END max_depth=6, min_samples_leaf=1, min_samples_split=2, n_estimators=200;, score=0.922 total time=   0.2s
[CV 2/3] END max_depth=6, min_samples_leaf=1, min_samples_split=2, n_estimators=200;, score=0.927 total time=   0.2s
[CV 3/3] END max_depth=6, min_samples_leaf=1, min_samples_split=2, n_estimators=200;, score=0.912 total time=   0.2s
[CV 1/3] END max_depth=6, min_samples_leaf=1, min_samples_split=2, n_estimators=300;, score=0.922 total time=   0.3s
[CV 2/3] END max_depth=6, min_samples_leaf=1, min_samples_split=2, n_estimators=300;, score=0.927 total time=   0.3s
[

GridSearchCV(cv=3,
             estimator=RandomForestClassifier(class_weight='balanced',
                                              random_state=42),
             param_grid={'max_depth': [6, 8, 10, 12],
                         'min_samples_leaf': [1, 2, 3, 4, 5],
                         'min_samples_split': [2, 3, 4, 5],
                         'n_estimators': [100, 200, 300, 400]},
             scoring='average_precision', verbose=3)

In [ ]:
model3 = CV.best_estimator_
show_result(model3)

Оптимальный порог: 0.4095
F1 Score: 0.4107142857142857
ROC AUC: 0.7157377896459644
PR AUC: 0.34231900006952876
Confusion matrix: 
 [[205  42]
 [ 24  23]]


In [ ]:
estimators = [
    ('CB', model),
    ('XGB', model2),
    ('RF', model3)
]
stack_model = StackingClassifier(
    estimators=estimators, final_estimator=CatBoostClassifier(loss_function='Logloss', auto_class_weights='Balanced', early_stopping_rounds=15, verbose=0, random_state=42)
).fit(x_train, y_train)
show_result(stack_model)

Оптимальный порог: 0.1277
F1 Score: 0.4195804195804196
ROC AUC: 0.7017830993194935
PR AUC: 0.3276791609718192
Confusion matrix: 
 [[181  66]
 [ 17  30]]


In [ ]:
import joblib

In [ ]:
import joblib

modelv1 = joblib.load('model_v1.pkl')
show_result(modelv1)

Оптимальный порог: 0.1636
F1 Score: 0.5242718446601942
ROC AUC: 0.7936084072702213
PR AUC: 0.5024290886174043
Confusion matrix: 
 [[218  29]
 [ 20  27]]
